# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shashank007-ux/Week-1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row represents one anonymized content item (one page or article) in the starter dataset. The data is summarized around the recent performance window, especially the prior 30-day and last 30-day impressions windows, so the grain is `content_id` and the time window is the page’s recent 30-day performance history.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from pathlib import Path
import pandas as pd


def load_refresh_data():
    base = Path.cwd()
    for candidate in [base, base.parent, base.parent.parent]:
        path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError("Could not find the starter dataset.")


df = load_refresh_data()
print("Rows, columns:", df.shape)
print("Unique content_id:", df["content_id"].nunique())
print("Unique client_id:", df["client_id"].nunique())
print("Sample window columns:")
print(df[["impressions_prev_30d", "impressions_last_30d", "impressions_90d"]].head())
print("No missing prev/last impressions:", df[["impressions_prev_30d", "impressions_last_30d"]].isna().sum().sum())

## 2. Fields: feature / label / context / excluded

Features:
- `search_volume`, `competition`, `competition_level`, `cpc`
- `content_type`, `main_intent`
- `word_count`, `char_count`, `content_age_days`, `days_since_last_update`
- `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`
- `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`
- `impression_tier`, `position_tier`, `age_tier`, `word_count_tier`, `char_count_tier`

Label:
- `impressions_last_30d < impressions_prev_30d` as a decline proxy for future opportunity scoring.

Context:
- `content_id`, `client_id` to maintain the item identity and grouping.
- `trend_direction`, `trend_pct` as additional descriptive status signals.

Excluded:
- `clicks_last_30d`, `sessions_last_30d` because they come from the same future/label window and should not be used as training features.
- `impressions_90d` when strict label alignment is required, since using a full 90-day summary can mix the training window with the target window.
- Any identifiers or fields that leak editorial actions or private content details (not present in this starter dataset).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path


def load_refresh_data():
    base = Path.cwd()
    for candidate in [base, base.parent, base.parent.parent]:
        path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError("Could not find the starter dataset.")


df = load_refresh_data()

print("Feature list count:", 35)
print("Context fields:", ["content_id", "client_id", "trend_direction", "trend_pct"])
print("Label proxy definition:", "impressions_last_30d < impressions_prev_30d")
print("Excluded future-window fields:", ["clicks_last_30d", "sessions_last_30d"])
print("Impression 90d field excluded when strict label-window alignment is needed.")

## 3. Verify it with queries (grain, counts, missing values, windows)
The grain is one `content_id` per row, and the dataset contains 30,000 unique content items. The time windows are represented by the `impressions_prev_30d` and `impressions_last_30d` columns, which are non-missing for all rows and support the decline proxy label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path


def load_refresh_data():
    base = Path.cwd()
    for candidate in [base, base.parent, base.parent.parent]:
        path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError("Could not find the starter dataset.")


df = load_refresh_data()

print("content_id unique:", df["content_id"].nunique())
print("rows:", len(df))
print("most common content_type:\n", df["content_type"].value_counts().head())
print("\nImpression window stats:")
print(df[["impressions_prev_30d", "impressions_last_30d"]].describe())
print("\nMissing trend_pct:", df["trend_pct"].isna().sum())
print("Trend direction values:\n", df["trend_direction"].value_counts())
print("\nOverlap check: prev/last windows seem defined by columns, not dates.")

## 4. Data limits

This data can tell us which pages were declining recently, but it cannot prove that a refresh caused later gains. It does not include content text, URLs, editorial decisions, or the actual refresh actions taken on a page. The strongest use is decision support: ranking candidate pages to review, not claiming causal impact or guaranteed recovery.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from pathlib import Path


def load_refresh_data():
    base = Path.cwd()
    for candidate in [base, base.parent, base.parent.parent]:
        path = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError("Could not find the starter dataset.")


df = load_refresh_data()

print("This data cannot tell us causal impact from refresh actions.")
print("It does not include content text, editorial decisions, or actual refresh events.")
print("It can tell us which pages were declining recently, but not whether a refresh would have fixed them.")
print("This is decision-support data, not an experiment or causal proof set.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.